In [1]:
import os
import json
import requests
from typing import Dict
from dotenv import load_dotenv
load_dotenv()

API_URL = "http://localhost:3000/api/search"

def call_api(query: str, history: list = None) -> Dict:
    if history is None:
        history = [
            ["human", "Hi, how are you?"],
            ["assistant", "I am doing well, how can I help you today?"]
        ]
    
    llm_provider, llm_name = os.getenv("PERPLEXITY_LLM_PROVIDER"), os.getenv("PERPLEXITY_LLM")
    embed_model_provider, embed_model_name_name = os.getenv("PERPLEXITY_EMBEDDING_PROVIDER"), os.getenv("PERPLEXITY_EMBEDDING_MODEL")
    
    
    headers = {"Content-Type": "application/json"}
    payload = {
        "chatModel": {
            "provider": llm_provider,
            "name": llm_name
        },
        "embeddingModel": {
            "provider": embed_model_provider,
            "name": embed_model_name_name
        },
        "optimizationMode": "speed",
        "focusMode": "webSearch",
        "query": query,
        "history": history
    }
    
    response = requests.post(API_URL, headers=headers, data=json.dumps(payload)).json()
    if response.get("error"):
        raise ValueError(f"API error: {response['error']}")
    return response

In [2]:
def get_competitors(company_name: str, n_competitors: int = 3) -> Dict:
    """
    Get competitors of a company using the API.
    
    Args:
        company_name (str): The name of the company.
        n_competitors (int): The number of competitors to retrieve.
        
    Returns:
        Dict: The API response containing competitors information.
    """
    if not isinstance(n_competitors, int) or n_competitors <= 0:
        raise ValueError("n_competitors must be a positive integer.")
    
    competitors_extraction_prompt = (
        f"Provide me top {n_competitors} competitors of {company_name}."
        "Make sure to include the competitors landing page URL, "
        "and the description of the company and why it is a competitor."
        "Use the landing page of the competitor to extract the description."
    )

    api_response = call_api(competitors_extraction_prompt)
    return api_response

In [ ]:
competitors_extraction_prompt = (
    "Provide me top 3 competitors of Whatfix."
    "Make sure to include the --\n"
    "1. competitors landing page URL, \n"
    "2. The description of the company\n"
    "3. why it is a competitor.\n"
    "Use the landing page of the competitor to extract the description.\n"
)

api_response = call_api(competitors_extraction_prompt)

In [4]:
print(api_response['message'])

Here are the top three competitors of Whatfix, along with their landing page URLs and descriptions extracted from their respective websites:

## 1. WalkMe
- **Landing Page URL**: [WalkMe](https://www.walkme.com)
- **Description**: WalkMe is a leading digital adoption platform that helps organizations simplify the user experience and drive user engagement. By providing interactive guidance and insights, WalkMe enables businesses to enhance their software adoption and improve overall productivity. Its robust features include step-by-step walkthroughs, analytics, and integrations with various applications, making it a strong competitor to Whatfix in the digital adoption space.

## 2. UserGuiding
- **Landing Page URL**: [UserGuiding](https://userguiding.com)
- **Description**: UserGuiding is a user onboarding tool designed to help businesses create interactive product tours and guides without any coding. It focuses on enhancing user experience by providing in-app guidance and feedback mech

In [1]:
search_engine_results = \
"""
Here are the top three competitors of Whatfix, along with their landing page URLs and descriptions extracted from their respective websites:

## 1. WalkMe
- **Landing Page URL**: [WalkMe](https://www.walkme.com)
- **Description**: WalkMe is a leading digital adoption platform that helps organizations simplify the user experience and drive user engagement. By providing interactive guidance and insights, WalkMe enables businesses to enhance their software adoption and improve overall productivity. Its robust features include step-by-step walkthroughs, analytics, and integrations with various applications, making it a strong competitor to Whatfix in the digital adoption space.

## 2. UserGuiding
- **Landing Page URL**: [UserGuiding](https://userguiding.com)
- **Description**: UserGuiding is a user onboarding tool designed to help businesses create interactive product tours and guides without any coding. It focuses on enhancing user experience by providing in-app guidance and feedback mechanisms. UserGuiding's ease of use and affordability make it an attractive alternative to Whatfix, particularly for companies looking for straightforward onboarding solutions.

## 3. Appcues
- **Landing Page URL**: [Appcues](https://www.appcues.com)
- **Description**: Appcues is a product experience platform that allows teams to create personalized onboarding experiences and feature announcements without needing engineering resources. With its no-code interface, Appcues empowers product managers and marketers to enhance user engagement and retention. Its focus on user experience and customization positions it as a significant competitor to Whatfix, especially for businesses aiming to improve their product adoption strategies.

These competitors offer various features and benefits that cater to different organizational needs, making them viable alternatives to Whatfix in the digital adoption platform market.
"""

In [2]:
from pydantic import BaseModel
from echo.echo_agent import get_crew


class Competitor(BaseModel):
    name: str
    description: str
    url: str
    rationale: str

class CompetitorsExtractionResponse(BaseModel):
    competitors: list[Competitor]

   
agent_templates = {
    "CompetitorExtractionAgent": dict(
        role="Competitor Research Agent",
        goal=(
            "You are an expert in extracting out the list of competitors of a sales company."
        ),
        backstory=(
            "A sales company is trying to sell its product to a customer."
            "The customer is asking for a list of competitors of the sales company."
            "You are an expert in extracting out the list of competitors of a sales company."
        ),
    )
}

task_templates = {
    "CompetitorsExtractionTask": dict(
        name='Competitors Extraction',
        description=(
            "Given the following result from a search engine, extract the competitors of the company."
            "Make sure to include the competitors landing page URL, and the description of the company and why it is a competitor."
            "Below is the search engine result -\n"
            "{search_engine_result}\n"
        ),
        expected_output=(
            "A list of .\n"
            "The response should conform to the provided schema.\n"
            "You need to extract the following information in the following pydantic structure -\n"
            "{pydantic_structure}\n"
            "Make sure there are no comments in the response JSON and it should be a valid JSON."
        ),
        output_pydantic=CompetitorsExtractionResponse,
        agent="CompetitorExtractionAgent",
    )   
}

crew = get_crew(
    agent_templates=agent_templates,
    task_templates=task_templates,
)

In [3]:
response = crew.kickoff(
    inputs={
        "search_engine_result": search_engine_results
    }
)

In [4]:
from typing import List


competitors: List[Competitor] = response.tasks_output[0].pydantic.competitors


In [5]:
import asyncio
import nest_asyncio
from echo.tools.web_scraping import extract_data_from_website
nest_asyncio.apply()

competitors_website_data = {}
for competitor in competitors:
    print(f"Extracting data from {competitor.url}")
    website_data = asyncio.run(extract_data_from_website(competitor.url))
    competitors_website_data[competitor.name] = website_data

Extracting data from https://www.walkme.com


Extracting Navigation Links:   0%|          | 0/8 [00:00<?, ?it/s]

Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed


Getting final links
Final Links: ['https://www.walkme.com/why-walkme/', 'https://www.walkme.com/digital-adoption-platform/', 'https://www.walkme.com/solutions/', 'https://www.walkme.com/pricing/', 'https://www.walkme.com/customer-stories/', 'https://www.walkme.com/resource-library/', 'https://www.walkme.com/blog/', 'https://www.walkme.com/digital-adoption-institute/', 'https://www.walkme.com/center-of-excellence/', 'https://www.walkme.com/resources/white-papers/walkme-delivers-a-3-year-494-roi/', 'https://www.walkme.com/what-is-digital-adoption/', 'https://www.walkme.com/what-is-digital-transformation/', 'https://www.walkme.com/request-a-demo/', 'https://www.walkme.com/events/', 'https://www.walkme.com/newsroom/']
Processing link: https://www.walkme.com/solutions/
Processing link: https://www.walkme.com/resource-library/
Processing link: https://www.walkme.com/digital-adoption-platform/
Processing link: https://www.walkme.com/blog/
Processing link: https://www.walkme.com/customer-stori

Summarizing Data From Links:   0%|          | 0/15 [00:00<?, ?it/s]

...

Extracting Navigation Links:   0%|          | 0/11 [00:00<?, ?it/s]

Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed


Getting final links
Final Links: ['https://www.appcues.com/product', 'https://www.appcues.com/pricing', 'https://www.appcues.com/how-it-works', 'https://www.appcues.com/why-appcues', 'https://www.appcues.com/customers', 'https://www.appcues.com/resources', 'https://www.appcues.com/use-cases', 'https://www.appcues.com/use-case/onboarding', 'https://www.appcues.com/use-case/free-trial-conversion', 'https://www.appcues.com/use-case/feature-adoption', 'https://www.appcues.com/use-case/feedback', 'https://www.appcues.com/use-case/support', 'https://www.appcues.com/integrations', 'https://www.appcues.com/resource/product-adoption-academy', 'https://www.appcues.com/made-with-appcues', 'https://www.appcues.com/blog', 'https://www.productled.org/']
Processing link: https://www.appcues.com/use-case/onboarding
Processing link: https://www.appcues.com/pricing
Processing link: https://www.appcues.com/product
Processing link: https://www.appcues.com/use-cases
Processing link: https://www.appcues.com

Summarizing Data From Links:   0%|          | 0/15 [00:00<?, ?it/s]

Failed to fetch page, status code: 404Failed to fetch page, status code: 404

Failed to fetch page, status code: 404
Failed to extract text from https://www.appcues.com/use-cases
Processing link: https://www.appcues.com/use-case/feedback
Failed to extract text from https://www.appcues.com/resources
Processing link: https://www.appcues.com/why-appcues
Failed to extract text from https://www.appcues.com/resource/product-adoption-academy
Processing link: https://www.appcues.com/how-it-works
Extracted data from https://www.appcues.com/made-with-appcues
Processing link: https://www.appcues.com/integrations
Extracted data from https://www.appcues.com/use-case/support
Processing link: https://www.appcues.com/use-case/free-trial-conversion
Failed to fetch page, status code: 404
Failed to extract text from https://www.appcues.com/use-case/free-trial-conversion
Extracted data from https://www.appcues.com/use-case/onboarding
Extracted data from https://www.appcues.com/pricing
Extracted data from 

In [7]:
competitor

Competitor(name='Appcues', description='Appcues is a product experience platform that allows teams to create personalized onboarding experiences and feature announcements without needing engineering resources.', url='https://www.appcues.com', rationale='Its focus on user experience and customization positions it as a significant competitor to Whatfix, especially for businesses aiming to improve their product adoption strategies.')

In [11]:
{
    **competitor.model_dump(mode="json"),
    "website_data": competitors_website_data[competitor.name],
}

{'name': 'Appcues',
 'description': 'Appcues is a product experience platform that allows teams to create personalized onboarding experiences and feature announcements without needing engineering resources.',
 'url': 'https://www.appcues.com',
 'rationale': 'Its focus on user experience and customization positions it as a significant competitor to Whatfix, especially for businesses aiming to improve their product adoption strategies.',
 'website_data': 'Link: https://www.appcues.com/made-with-appcues\nData: **Title:** "How Appcues Drives User Engagement and Product Adoption for SaaS Businesses"\n\n**Summary:**  \nAppcues is an all-in-one user engagement platform trusted by over 1,500 scaling SaaS businesses. It specializes in driving product adoption and user engagement through a suite of tools designed to enhance the customer experience. Key features include:\n\n- **In-app Messaging:** Deliver native-looking, contextual messages directly within your app to guide users effectively.  \n